# Real-Time Voice AI

**Module:** 16 — Speech AI

Latency budgets, transport, speculative responses, and partials for duplex voice.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Build a numeric latency budget for a voice turn
- Compare WebSocket, WebRTC, and telephony transports
- Apply speculative and partial-response tactics
- Measure time-to-first-audio and barge-in responsiveness


## Latency Budget

### Definition
A latency budget allocates milliseconds across VAD, ASR, LLM, TTS, and network so the turn feels alive.

### Why it matters
Users perceive delays >~700–1000ms as awkward gaps in conversation.

### How it works
Write budgets; stream every stage; overlap LLM+TTS; cut prompt tokens; cache FAQs; prefer regional endpoints.

### Intuition
A relay race — handoffs must be rehearsed.

### Pitfalls
- Optimizing only LLM TTFT while ASR endpoints slowly
- Gigantic system prompts on every turn

### When to use
All conversational voice products.


### Example budget (conversational)

| Stage | Target ms | Tactic |
|-------|-----------|--------|
| Endpointing | 150–300 | Aggressive but not rude |
| ASR final/partial | 100–300 | Streaming ASR |
| LLM first token | 200–400 | Small model / cache |
| TTS first audio | 100–250 | Streaming synth |
| Network | 50–150 | Regional edge |
| **Total** | **~600–900** | Overlap stages |

```mermaid
gantt
  title Turn timeline (illustrative)
  dateFormat X
  axisFormat %L
  section User
  Speaking           :0, 1200
  section System
  Partials ASR       :800, 400
  LLM stream         :1250, 350
  TTS stream         :1400, 500
```


In [ ]:
# Demo 1: budget calculator
from dataclasses import dataclass

@dataclass
class Budget:
    endpoint: int = 200
    asr: int = 250
    llm: int = 350
    tts: int = 180
    net: int = 100
    def total(self): return self.endpoint + self.asr + self.llm + self.tts + self.net
    def with_overlap(self):
        # LLM starts on stable partial; TTS overlaps LLM
        return self.endpoint + max(self.asr, 120) + max(self.llm, self.tts) + self.net

b = Budget()
print("serial", b.total(), "overlapped", b.with_overlap())


In [ ]:
# Demo 2: speculative FAQ responder
FAQS = {"hours": "We're open 9 to 5 local time.", "reset": "To reset your password, open settings, then security."}

def speculative(partial: str):
    p = partial.lower()
    for k, ans in FAQS.items():
        if k in p:
            return {"speculative": True, "answer": ans}
    return {"speculative": False}

print(speculative("what are your hours of"))
print(speculative("I want to cancel"))


## Transport

### Definition
Transport carries audio frames bidirectionally with different latency/reliability tradeoffs.

### Why it matters
Bad transport dwarfs model improvements (jitter, packet loss, AEC issues).

### How it works
WebRTC for browsers/apps (AEC/jitter buffers); WebSockets for many agent SDKs; SIP/PSTN for phones.

### Intuition
The pipes matter as much as the pumps.

### Pitfalls
- Sending huge WAV blobs per turn over HTTP only
- No jitter buffer tuning on mobile

### When to use
Client design and partner telephony.


### Transport comparison

| Transport | Pros | Cons |
|-----------|------|------|
| WebRTC | Low latency, AEC, NAT traversal | Complexity |
| WebSocket PCM | Simple app integration | You own AEC/jitter |
| HTTP chunked | Easy batch-ish | Poor duplex |
| PSTN/SIP | Phone reach | 8 kHz, carrier quirks |


In [ ]:
# Demo 3: jitter buffer (toy)
from collections import deque

class JitterBuffer:
    def __init__(self, target=3):
        self.target = target
        self.q = deque()
    def push(self, seq, frame):
        self.q.append((seq, frame)); self.q = deque(sorted(self.q))
    def pop(self):
        if len(self.q) < self.target: return None
        return self.q.popleft()

jb = JitterBuffer()
for seq in (2, 0, 1, 3):
    jb.push(seq, f"f{seq}")
    print("pop", jb.pop())


In [ ]:
# Demo 4: time-to-first-audio metric
import time

def ttfa_ms(t_user_end, t_first_audio):
    return int((t_first_audio - t_user_end) * 1000)

t0 = time.time()
# simulate
t_user_end = t0 + 1.2
t_first = t0 + 1.2 + 0.65
print("TTFA", ttfa_ms(t_user_end, t_first), "ms")


## Speculative & Partial Responses

### Definition
Systems may start speaking or tool prep before the final user utterance / final LLM token.

### Why it matters
Cuts dead air; risks wrong starts — needs cancel on revision.

### How it works
Stable ASR partials → draft LLM; speculative FAQ; TTS on sentence boundaries; cancel on barge-in or hyp change.

### Intuition
A bartender pouring as you finish ordering — sometimes wrong drink.

### Pitfalls
- Never canceling speculative TTS
- Speculating irreversible tools

### When to use
Latency-sensitive agents with safe speculation.


In [ ]:
# Demo 5: sentence-boundary TTS flusher
import re

class TTSStreamer:
    def __init__(self):
        self.buf = ""
        self.spoken = []
    def push_llm(self, delta):
        self.buf += delta
        while True:
            m = re.search(r"(.+?[.!?])(\s|$)", self.buf)
            if not m: break
            sentence = m.group(1).strip()
            self.spoken.append(sentence)
            self.buf = self.buf[m.end():]
        return list(self.spoken)

s = TTSStreamer()
print(s.push_llm("Sure — I can help. "))
print(s.push_llm("Your order 4455 "))
print(s.push_llm("is canceled."))


In [ ]:
# Demo 6: cancel speculative path on hypothesis revise
class SpecState:
    def __init__(self):
        self.hyp = ""
        self.started = False
    def update(self, hyp):
        if self.started and not hyp.startswith(self.hyp[: max(1, len(self.hyp)//2)]):
            self.started = False
            return "cancel_speculation"
        self.hyp = hyp
        if len(hyp.split()) >= 4 and not self.started:
            self.started = True
            return "start_speculation"
        return "hold"

st = SpecState()
for h in ["I want", "I want to cancel", "actually keep my order"]:
    print(h, "->", st.update(h))


### Checklist — Realtime readiness

- [ ] Written latency budget
- [ ] TTFA dashboarded
- [ ] Transport chosen with AEC story
- [ ] Speculative paths cancellable
- [ ] Packet loss tested on mobile


### Try it yourself — Realtime

1. Rebalance Demo 1 budget for telephony (+net, +endpoint).
2. Add metrics labels: turn_id, stage, ms.
3. List 5 ways speculation can go wrong.

**Stretch:** Build a fake WebSocket message log for PCM chunks.


### Try it yourself — Transport

1. When would you pick WebRTC over WS?
2. Document PSTN μ-law resampling steps to 16 kHz ASR.


## Knowledge Check

**Q1.** What is TTFA?

<details><summary>Answer</summary>

Time-to-first-audio — milliseconds from user stop (or endpoint) to first audible bot audio.

</details>

**Q2.** Why overlap LLM and TTS?

<details><summary>Answer</summary>

Waiting for full LLM text before synthesizing adds avoidable dead air.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `TTFA` | Time to first audio |
| `WebRTC` | Realtime media transport stack |
| `jitter buffer` | Reorders/smoothes packets |
| `speculation` | Start work before finals |
| `AEC` | Acoustic echo cancellation |
| `duplex` | Talk and listen concurrently |


## Key Takeaways

- Budget milliseconds per stage and overlap
- Transport/AEC can dominate model latency
- Partials + speculative TTS cut dead air
- Always make speculation cancellable


## Production Incident Patterns — realtime voice

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "realtime voice",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — realtime voice

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("realtime voice", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — realtime voice ops

1. Draft an on-call runbook bullet list for realtime voice when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
